[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/11_sliding_window.ipynb)

# 🔴 Hard: Sliding Window Attention

Implement **Sliding Window Attention** — used in Longformer, Mistral, etc. for efficient long-context processing.

Each position $i$ can only attend to positions $j$ where $|i - j| \le w$ (the window size).

### Signature
```python
def sliding_window_attention(Q, K, V, window_size):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
    # window_size: int — position i attends to [i-w, i+w]
```

### Rules
- Do **NOT** use sparse attention libraries
- Mask positions outside the window with `-inf`
- `window_size=0`: only self — output should equal V
- `window_size >= seq_len`: equivalent to full attention

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.0 MB/s eta 0:00:00


In [2]:
import torch
import math

In [13]:
# ✏️ YOUR IMPLEMENTATION HERE

def sliding_window_attention(Q, K, V, window_size):
    from torch.nn.functional import scaled_dot_product_attention
    seq_len = Q.shape[1]
    if window_size == 0:
      return V
    elif window_size >= seq_len:
      return scaled_dot_product_attention(Q, K, V)
    else:
      bsz, seq_len, _ = Q.shape
      attn_mask = torch.ones(seq_len, seq_len, dtype=torch.bool)
      attn_mask = torch.tril(attn_mask)
      attn_mask = torch.triu(attn_mask, diagonal=-window_size)
      return scaled_dot_product_attention(Q, K, V, attn_mask=attn_mask)

In [14]:
# 🧪 Debug
Q = torch.randn(1, 6, 8)
K = torch.randn(1, 6, 8)
V = torch.randn(1, 6, 8)

out = sliding_window_attention(Q, K, V, window_size=1)
print("Output shape:", out.shape)  # (1, 6, 8)

# window=0 should return V
out0 = sliding_window_attention(Q, K, V, window_size=0)
print("window=0 == V?", torch.allclose(out0, V, atol=1e-5))

Output shape: torch.Size([1, 6, 8])
window=0 == V? True


In [15]:
from torch_judge import check
check('sliding_window')


🧪 Testing: Sliding Window Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (3.1ms)
  ✅ [2/5] window_size=0 — only sees itself (0.5ms)
  ✅ [3/5] Large window equals full attention (6.6ms)
  ✅ [4/5] Distant tokens don't affect output (3.0ms)
  ✅ [5/5] Gradient flow (1.4ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (14.6ms total)
  Progress saved. Run status() to see your dashboard.

